# Office Items Detection: YOLO Training & Experiments
**PDE3802 - Assessment 1**

**Goal:** Train a lightweight object detector that recognises everyday office items (bottle, laptop, mug, bin, mouse, smartphone, scissors).
Transfer learning was used from COCO-pretrained YOLO nano models (YOLOv8n and YOLO11n) in Ultralytics.

---
## 1. Setup

### 1.1 Environment
Clone the project repo (code + dataset) and install Ultralytics and Albumentations

In [ ]:
!git clone https://github.com/meganeMDX/PDE3802_Assessment1.git
%cd PDE3802_Assessment1
!pip install ultralytics
!pip install albumentations

Cloning into 'PDE3802_Assessment1'...
remote: Enumerating objects: 49667, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 49667 (delta 8), reused 10 (delta 4), pack-reused 49650 (from 1)
Receiving objects: 100% (49667/49667), 1.45 GiB | 19.28 MiB/s, done.
Resolving deltas: 100% (274/274), done.
Updating files: 100% (41694/41694), done.
/content/PDE3802_Assessment1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


### 1.2 Dataset validation
Check that `dataset/processed` has the expected split: 490 / 140 / 70 images per class for train / valid / test, with 7 classes (3,430 / 980 / 490 images).

In [ ]:
from pathlib import Path

# Validate the processed dataset folder
DATASET_DIR = Path("/content/PDE3802_Assessment1/dataset/processed")

EXPECTED_COUNTS = {
    "train": 7 * 490,
    "valid": 7 * 140,
    "test": 7 * 70
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

all_valid = True

for split, expected_count in EXPECTED_COUNTS.items():
    split_dir = DATASET_DIR / split
    images_dir = split_dir / "images"

    if not split_dir.is_dir():
        print(f"Missing folder: {split_dir}")
        all_valid = False
        continue

    if not images_dir.is_dir():
        print(f"Missing images folder: {images_dir}")
        all_valid = False
        continue

    image_count = sum(
        1 for file in images_dir.iterdir()
        if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
    )

    if image_count == expected_count:
        print(f"{split}: {image_count} images")
    else:
        print(f"{split}: Expected {expected_count}, found {image_count}")
        all_valid = False

if not all_valid:
    raise RuntimeError("Dataset validation failed. Check the dataset before training.")

print("\nDataset validation successful.")

train: 3430 images
valid: 980 images
test: 490 images

Dataset validation successful.


### 1.3 Shared configuration
Paths and imports used by every experiment below. Run this cell first after any Colab restart.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

ROOT = Path("/content/PDE3802_Assessment1")
CONFIG_FILE = ROOT / "dataset" / "data.yaml"

---
## 2. Experiments

Experiments are grouped into four phases. The first three follow the dataset as it improved and Phase 4 is hyperparameter tuning on the final dataset.

### Phase 1: Initial dataset (A) - YOLOv8n vs YOLO11n

#### Experiment 1.1: YOLOv8n baseline
An initial YOLOv8n baseline was trained with the repo's `src/train.py` (Ultralytics defaults, 100 epochs) as the reference point.

In [ ]:
# Experiment 1.1: YOLOv8n baseline (runs src/train.py)
!python /content/PDE3802_Assessment1/src/train.py

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.189 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None

**Result:** mAP50 **0.776**, mAP50-95 **0.594** (P 0.808, R 0.740).

---



#### Experiment 1.2: Hyperparameter tuning on YOLOv8n
`model.tune` was used on YOLOv8n (10 iterations × 30 epochs, AdamW) over lr0, lrf, momentum, weight_decay, warmup_epochs, hsv_s, hsv_v, translate, scale and mosaic.

In [ ]:
# Experiment 1.2: tuning with yolov8n.pt
search_space = {
    # optimisation
    "lr0": (0.0001, 0.003),
    "lrf": (0.01, 0.1),
    "momentum": (0.85, 0.98),
    "weight_decay": (0.00001, 0.001),
    "warmup_epochs": (0.0, 5.0),

    # data augmentation
    "hsv_s": (0.3, 0.9),
    "hsv_v": (0.2, 0.6),
    "translate": (0.0, 0.15),
    "scale": (0.3, 0.7),
    "mosaic": (0.5, 1.0),
}

model = YOLO("yolov8n.pt")

model.tune(
    data=CONFIG_FILE,
    epochs=30,
    iterations=10,
    imgsz=640,
    batch=16,
    patience=10,
    optimizer="AdamW",
    seed=42,
    space=search_space,
    name="office_items_detection",
    plots=True,
    save=True,
    val=True,
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Tuner: Initialized Tuner instance with 'tune_dir=/content/PDE3802_Assessment1/runs/detect/office_items_tuningTest'
Tuner: 💡 Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/10 with hyperparameters: {'lr0': 0.003, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'hsv_s': 0.7, 'hsv_v': 0.4, 'translate': 0.1, 'scale': 0.5, 'mosaic': 1.0}

MultiTrainer 1/1: fine-tuning on /content/PDE3802_Assessment1/dataset/data.yaml
Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, 

**Result:** best fitness (mAP50-95 after 30 epochs) **0.548**, reached at iteration 3 (iteration 1 scored 0.522).

Best settings: `lr0=0.002, lrf=0.01, momentum=0.96567, weight_decay=0.0003, warmup_epochs=2.38283, hsv_s=0.82138, hsv_v=0.45851, translate=0.10773, scale=0.3865, mosaic=0.9774`.

#### Experiment 1.3: YOLOv8n with the tuned hyperparameters
YOLOv8n was retrained for 100 epochs using the best settings from Experiment 1.2.

In [ ]:
# Experiment 1.3: YOLOv8n trained with the best hyperparameters from 1.2
model = YOLO("yolov8n.pt")

results = model.train(
    data=CONFIG_FILE,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    optimizer="AdamW",
    seed=42,

    # optimisation
    lr0=0.002,
    lrf=0.01,
    momentum=0.96567,
    weight_decay=0.0003,
    warmup_epochs=2.38283,

    # data augmentation
    hsv_s=0.82138,
    hsv_v=0.45851,
    translate=0.10773,
    scale=0.3865,
    mosaic=0.9774,

    # save results
    name="office_items_tuned_model",
    project=ROOT / "runs" / "detect",
    plots=True,
    save=True,
    val=True,
)

print("Training complete.")
print(f"Best model: {results.save_dir / 'weights' / 'best.pt'}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/PDE3802_Assessment1/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscri

**Result:** mAP50 **0.756**, mAP50-95 **0.570**, which is *worse* than the baseline.

**Observation:** The tuned settings did not transfer to the full training run. They use a much higher learning rate (`lr0=0.002` vs the 0.0008 that `optimizer=auto` chose for the baseline). That was likely rewarded in the short 30-epoch search but does not pay off over 100 epochs. The search was also small (10 iterations).

#### Experiment 1.4: YOLO11n baseline
Same parameters were used as 1.1, but with YOLO11n, to compare the two architectures on identical data.

In [ ]:
# Experiment 1.4: YOLO11n baseline (transfer learning)
model = YOLO("yolo11n.pt")

results = model.train(
    data=CONFIG_FILE,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    seed=42,
    name="yolo11n_baseline",
    project=ROOT / "runs" / "detect",
    plots=True,
    save=True,
    val=True,
)

print("Training complete.")
print(f"Best model: {results.save_dir / 'weights' / 'best.pt'}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/PDE3802_Assessment1/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscri

**Result:** mAP50 **0.790**, mAP50-95 **0.614** (P 0.792, R 0.746).

**Observation:** YOLO11n performs better than YOLOv8n on the same data (+0.014 mAP50, +0.020 mAP50-95). Later experiments mostly use YOLO11n.

Switching architecture helped more than tuning (Experiment 1.3 actually degraded performance).

### Phase 2: Expanded dataset (B) with more images and larger input size

#### Experiment 2.1: YOLOv8n baseline on the larger dataset
The YOLOv8n baseline was retrained on dataset B (grown to 700 images per class in total).

In [ ]:
# Experiment 2.1: second YOLOv8n baseline on the expanded dataset (700 images per class)
model = YOLO("yolov8n.pt")

results = model.train(
    data=CONFIG_FILE,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    seed=42,
    name="yolov8n_second_baseline",
    project=ROOT / "runs" / "detect",
    plots=True,
    save=True,
    val=True,
)

print("Training complete.")
print(f"Best model: {results.save_dir / 'weights' / 'best.pt'}")

Ultralytics 8.4.194 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/PDE3802_Assessment1/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_second_baseline, nbs=64, nms=None,

**Result:** mAP50 **0.863**, mAP50-95 **0.675** (P 0.873, R 0.803).

**Observation:** Better than Experiment 1.1 (+0.087 / +0.081), but the weakest classes are `phone` (0.745 mAP50) and `laptop` (0.812).

#### Experiment 2.2: YOLO11n at a larger input size (800)
YOLO11n was trained at `imgsz=800` instead of 640 on dataset B, to see whether higher input resolution helps.

In [ ]:
# Experiment 2.2: YOLO11n with increased image size (800) on the expanded dataset
model = YOLO("yolo11n.pt")

results = model.train(
    data=CONFIG_FILE,
    epochs=100,
    imgsz=800,
    batch=16,
    patience=20,
    seed=42,
    name="yolo11n_third_baseline",
    project=ROOT / "runs" / "detect",
    plots=True,
    save=True,
    val=True,
)

print("Training complete.")
print(f"Best model: {results.save_dir / 'weights' / 'best.pt'}")

Ultralytics 8.4.194 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/PDE3802_Assessment1/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_third_baseline, nbs=64, nms=None, 

**Result:** mAP50 **0.872**, mAP50-95 **0.678** (P 0.864, R 0.823). Early-stopped at epoch 90 (best epoch 70).

**Observation:** only +0.009 mAP50 / +0.003 mAP50-95 over Experiment 2.1. While the increased image size helped for smaller objects, it degraded the performance for most of the other classes.

### Phase 3: Revised dataset

#### Experiment 3.1: YOLO11n baseline on revised dataset
YOLO11n was retrained on revised datasets and imgsz was set back to 640 to improve performance.

In [ ]:
# Experiment 3.1: YOLO11n on the revised dataset, image size back to 640
model = YOLO("yolo11n.pt")

results = model.train(
    data=CONFIG_FILE,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    seed=42,
    name="yolo11n_fourth_baseline",
    project=ROOT / "runs" / "detect",
    plots=True,
    save=True,
    val=True,
)

print("Training complete.")
print(f"Best model: {results.save_dir / 'weights' / 'best.pt'}")

Ultralytics 8.4.195 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/PDE3802_Assessment1/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_fourth_baseline, nbs=64, nms=None,

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Result:** mAP50 **0.889**, mAP50-95 **0.703** (P 0.913, R 0.826). This is the best validation result of all runs.

**Observation:** the weakest classes are `laptop` (mAP50 0.788, recall 0.671), then `mouse` and `bottle` (mAP50-95 0.59).

#### Experiment 3.2: New laptop dataset
The laptop images were replaced with a new laptop dataset and re-trained, to try to lift the weakest class.

In [ ]:
# Experiment 3.2: YOLO11n with the new laptop dataset
model = YOLO("yolo11n.pt")

results = model.train(
    data=CONFIG_FILE,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    seed=42,
    name="yolo11n_fifth_baseline",
    project=ROOT / "runs" / "detect",
    plots=True,
    save=True,
    val=True,
)

print("Training complete.")
print(f"Best model: {results.save_dir / 'weights' / 'best.pt'}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.197 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/PDE3802_Assessment1/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscrip

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Result:** mAP50 **0.876**, mAP50-95 **0.698** (P 0.873, R 0.822).

**Observation:** `laptop` remains the weakest class according to validation results.

#### Experiment 3.3: Re-split dataset
Training was done again after the dataset was re-split to exactly 490 / 140 / 70 images per class.

In [ ]:
# Experiment 3.3: YOLO11n on the re-split dataset (Baseline for Phase 4)
model = YOLO("yolo11n.pt")

results = model.train(
    data=CONFIG_FILE,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    seed=42,
    name="yolo11n_sixth_baseline",
    project=ROOT / "runs" / "detect",
    plots=True,
    save=True,
    val=True,
)

print("Training complete.")
print(f"Best model: {results.save_dir / 'weights' / 'best.pt'}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.197 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/PDE3802_Assessment1/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscrip

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Result:** mAP50 **0.874**, mAP50-95 **0.697** (P 0.890, R 0.815).

**Observation:** almost identical to Experiment 3.2 (same 980 validation images / 1,422 instances). Yet `laptop` mAP50 still moved from 0.717 to 0.691.

### Phase 4: Hyperparameter tuning on the final dataset using YOLO11n + custom augmentation

#### 4.0 Shared tuning configuration
- The search space is narrower than in Phase 1: lr0, lrf, weight_decay, degrees, translate, scale and mosaic.
- Custom Albumentations were added after experimentation with breaking.py: a blur augmentation (Gaussian / motion / box blur, applied with 20% probability).
- Tuning output goes to Google Drive so the search can be resumed after a Colab session reset.

In [ ]:
from google.colab import drive
import albumentations as A

drive.mount("/content/drive")

TUNING_ROOT = Path("/content/drive/MyDrive/PDE3802/tuning")
RUN_NAME = "office_items_tuning"
TUNING_ROOT.mkdir(parents=True, exist_ok=True)

# define ranges
search_space = {
    # optimisation
    "lr0": (0.0001, 0.003),
    "lrf": (0.01, 0.1),
    "weight_decay": (0.00001, 0.001),

    # data augmentation
    "degrees": (0.0, 15.0),
    "translate": (0.0, 0.15),
    "scale": (0.3, 0.7),
    "mosaic": (0.5, 1.0),
}

# custom augmentations
BLUR_PROBABILITY = 0.20

blur_augmentations = A.OneOf(
    [
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        A.MotionBlur(blur_limit=(3, 7), p=1.0),
        A.Blur(blur_limit=(3, 7), p=1.0),
    ],
    p=BLUR_PROBABILITY,
)

custom_augmentations = [
    A.to_dict(blur_augmentations),
    A.to_dict(A.MedianBlur(blur_limit=3, p=0.01)),
    A.to_dict(A.ToGray(p=0.01)),
    A.to_dict(A.CLAHE(p=0.01)),
]

#### Experiment 4.1: Tuning YOLO11n (10 iterations × 30 epochs)
The Colab session was cut during iteration 9, so the run was **resumed** from that iteration in a new session (second cell, `resume=True`).

In [ ]:
# Experiment 4.1a: tune YOLO11n (iterations 1-8 completed before the session was cut)
model = YOLO("yolo11n.pt")

model.tune(
    data=CONFIG_FILE,
    epochs=30,
    iterations=10,
    imgsz=640,
    batch=16,
    patience=10,
    optimizer="AdamW",
    seed=42,
    space=search_space,
    augmentations=custom_augmentations,
    project=TUNING_ROOT,
    name=RUN_NAME,
    plots=True,
    save=True,
    val=True,
)

Tuner: Initialized Tuner instance with 'tune_dir=/content/drive/MyDrive/PDE3802/tuning/office_items_tuning'
Tuner: 💡 Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/10 with hyperparameters: {'lr0': 0.003, 'lrf': 0.01, 'weight_decay': 0.0005, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'mosaic': 1.0}

MultiTrainer 1/1: fine-tuning on /content/PDE3802_Assessment1/dataset/data.yaml
Ultralytics 8.4.199 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[{'__version__': '2.0.8', 'transform': {'__class_fullname__': 'OneOf', 'p': 0.2, 'transforms': [{'__class_fullname__': 'GaussianBlur', 'p': 1.0, 'blur_limit': (3, 7), 'sigma_limit': (0.5, 3.0)}, {'__class_fullname__': 'MotionBlur', 'p': 1.0, 'allow_shifted': True, 'angle_range': (0.0, 360.0), 'blur_limit': (3, 7), 'direction_range': (-1.0, 1.0)}, {'__class_fullname__': 'Blur', 'p':

In [ ]:
# Experiment 4.1b: resume the interrupted tuning run from iteration 9
model = YOLO("yolo11n.pt")

model.tune(
    data=CONFIG_FILE,
    epochs=30,
    iterations=10,
    imgsz=640,
    batch=16,
    patience=10,
    optimizer="AdamW",
    seed=42,
    space=search_space,
    augmentations=custom_augmentations,
    project=TUNING_ROOT,
    name=RUN_NAME,
    plots=True,
    save=True,
    val=True,
    resume=True,
)

Tuner: Initialized Tuner instance with 'tune_dir=/content/drive/MyDrive/PDE3802/tuning/office_items_tuning'
Tuner: 💡 Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Resuming tuning run /content/drive/MyDrive/PDE3802/tuning/office_items_tuning from iteration 9...
Tuner: Starting iteration 9/10 with hyperparameters: {'lr0': 0.003, 'lrf': 0.01938, 'weight_decay': 0.00033, 'degrees': 2.97061, 'translate': 0.11144, 'scale': 0.52719, 'mosaic': 0.99286}

MultiTrainer 1/1: fine-tuning on /content/PDE3802_Assessment1/dataset/data.yaml
Ultralytics 8.4.199 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[{'__version__': '2.0.8', 'transform': {'__class_fullname__': 'OneOf', 'p': 0.2, 'transforms': [{'__class_fullname__': 'GaussianBlur', 'p': 1.0, 'blur_limit': (3, 7), 'sigma_limit': (0.5, 3.0)}, {'__class_fullname__': 'MotionBlur', 'p': 1.0, 'allow_shifted': 

**Result:** best fitness **0.648** at iteration 7 (P 0.810, R 0.788, mAP50 0.834, mAP50-95 0.648). Total ≈ 6 h of T4 time across both sessions.

Best settings: `lr0=0.003, lrf=0.01, weight_decay=0.00033, degrees=0.0, translate=0.08996, scale=0.52457, mosaic=0.99286`. `lr0` landed on the upper edge of its search range.

**Notes:**
- Tuning fitness is measured after only 30 epochs, so it is **not comparable** to the 100-epoch scores above.

#### Experiment 4.2: Final training with adjusted tuned parameters (`lr0=0.001`)
YOLO11n was trained again for 100 epochs with the tuned settings and the custom augmentations, adjusting two values from the tuner's best: `lr0` lowered from 0.003 to 0.001 and `degrees` set to 10.

In [ ]:
# Experiment 4.2: final training with adjusted tuned parameters (lr0 = 0.001)
best_hyp = {
    "lr0": 0.001,
    "lrf": 0.01,
    "weight_decay": 0.00033,
    "degrees": 10.0,
    "translate": 0.08996,
    "scale": 0.52457,
    "mosaic": 0.99286,
}

model = YOLO("yolo11n.pt")

model.train(
    data=CONFIG_FILE,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    optimizer="AdamW",
    seed=42,
    augmentations=custom_augmentations,
    name="yolo11n_tuned_final",
    **best_hyp
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.200 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[{'__version__': '2.0.8', 'transform': {'__class_fullname__': 'OneOf', 'p': 0.2, 'transforms': [{'__class_fullname__': 'GaussianBlur', 'p': 1.0, 'blur_limit': (3, 7), 'sigma_limit': (0.5, 3.0)}, {'__class_fullname__': 'MotionBlur', 'p': 1.0, 'allow_shifted': True, 'angle_range': (0.0, 360.0), 'blur_limit': (3, 7), 'direction_range': (-1.0, 1.0)}, {'__class_fullname__': 'Blur', 'p': 1.0, 'blur_limit': (3, 7)}]}}, {'__version__': '2.0.8', 'transform': {'__class_fullname__': 'MedianBlur', 'p': 0.01, 'blur_limit': (3, 3)}}, {'__version_

**Result:** mAP50 **0.875**, mAP50-95 **0.674** (P 0.871, R 0.815)

#### Experiment 4.3: Final training with `lr0=0.000909`
Same as 4.2 but with `lr0=0.000909`, the learning rate `optimizer=auto` picked for the baseline.

In [ ]:
# Experiment 4.3: final training with adjusted tuned parameters (lr0 = 0.000909)
best_hyp = {
    "lr0": 0.000909,
    "lrf": 0.01,
    "weight_decay": 0.00033,
    "degrees": 10.0,
    "translate": 0.08996,
    "scale": 0.52457,
    "mosaic": 0.99286,
}

model = YOLO("yolo11n.pt")

model.train(
    data=CONFIG_FILE,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    optimizer="AdamW",
    seed=42,
    augmentations=custom_augmentations,
    name="yolo11n_tuned_final",
    **best_hyp
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.201 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[{'__version__': '2.0.8', 'transform': {'__class_fullname__': 'OneOf', 'p': 0.2, 'transforms': [{'__class_fullname__': 'GaussianBlur', 'p': 1.0, 'blur_limit': (3, 7), 'sigma_limit': (0.5, 3.0)}, {'__class_fullname__': 'MotionBlur', 'p': 1.0, 'allow_shifted': True, 'angle_range': (0.0, 360.0), 'blur_limit': (3, 7), 'direction_range': (-1.0, 1.0)}, {'__class_fullname__': 'Blur', 'p': 1.0, 'blur_limit': (3, 7)}]}}, {'__version__': '2.0.8', 'transform': {'__class_fullname__': 'MedianBlur', 'p': 0.01, 'blur_limit': (3, 3)}}, {'__version_

**Result:** mAP50 **0.862**, mAP50-95 **0.673** (P 0.879, R 0.794).

**Phase 4 observation:** both tuned variants match the baseline on mAP50 (0.875 / 0.862 vs 0.874) but are about 0.02 lower on mAP50-95 (0.674 / 0.673 vs 0.697). Detection is not worse, but the boxes fit less tightly.

---
## 3. Conclusion

The **YOLO11n baseline** (Experiment 3.3) is the selected final model. It has a better overall performance compared to the tuned variants.